# Steps
1. Convert token và lemma raw text to csv to get the index
2. Create a list of lemma_POS
3. Extract sentences with lemma_POS in token and lemma csv into individual files for each lemma_POS
4. Parse lại token bằng Stanza
5. Check lại giữa tag cũ và mới xem tỉ lệ sai POS là bao nhiêu

## Import

In [ ]:
import pandas as pd
import os
import stanza
import re
from tqdm import tqdm
from pathlib import Path
import shutil
import sys
sys.path.append('../data_preprocessing')
from utils import open_txt, save_to_txt, search_in_txt, replace_in_txt, return_stanza_parsed_tags

In [ ]:
# Helper functions

# Convert from orginal pos to stanza
def convert_org_stanza(org_lemma_pos):

    org_lemma = org_lemma_pos.rsplit('_')[0]
    org_pos = org_lemma_pos.split('_')[-1]

    stanza_lemma = org_lemma
    if org_pos == 'nn':
        stanza_pos = 'NOUN'
    elif org_pos == 'vb':
        stanza_pos = 'VERB'
    
    return stanza_lemma, stanza_pos

## Convert token and lemma raw text to csv to get the index

In [ ]:
corp_no = 2
data_types = ['token', 'lemma']

In [ ]:
for data_type in data_types:
    raw_file = f'./SemEval/semeval2020_ulscd_eng/corpus{corp_no}/{data_type}/ccoha{corp_no}.txt'
    raw_csv_path = f'./SemEval/semeval2020_ulscd_eng/corpus{corp_no}/{data_type}/ccoha{corp_no}.csv'

    # Đọc file txt
    with open(raw_file, 'r') as f:
        lines = f.readlines()
        lines = [line.rstrip('\n') for line in lines]
        # lines = [line.strip() for line in lines if line.strip()]  # Remove empty lines
        # Save to DataFrame and then to CSV
        df = pd.DataFrame(lines)
        df.columns = ['sent']
        df.to_csv(raw_csv_path, index=False, header=True)

## Create a list of lemma_POS

In [ ]:
selected_lemmas = ['attack_nn', 'bag_nn', 'ball_nn', 'bit_nn', 'chairman_nn', 'circle_vb', 'contemplation_nn', 'donkey_nn', 'edge_nn', 'face_nn', 'fiction_nn', 'gas_nn', 'graft_nn', 'head_nn', 'land_nn', 'lane_nn', 'lass_nn', 'multitude_nn', 'ounce_nn', 'part_nn', 'pin_vb', 'plane_nn', 'player_nn', 'prop_nn', 'quilt_nn', 'rag_nn', 'record_nn', 'relationship_nn', 'risk_nn', 'savage_nn', 'stab_nn', 'stroke_vb', 'thump_nn', 'tip_vb', 'tree_nn', 'twist_nn', 'word_nn']

## Sem-Eval Faithful (split)

### Extract sentences with lemma_POS in token and lemma csv into individual files for each lemma_POS

In [ ]:
lemma_csv = f'./SemEval/semeval2020_ulscd_eng/corpus{corp_no}/lemma/ccoha{corp_no}.csv'
token_csv = f'./SemEval/semeval2020_ulscd_eng/corpus{corp_no}/token/ccoha{corp_no}.csv'

df_lemma = pd.read_csv(lemma_csv)
df_token = pd.read_csv(token_csv)

In [ ]:
out_folder_lemma = f'./SemEval_en_split/corpus{corp_no}/lemma/'
out_folder_token = f'./SemEval_en_split/corpus{corp_no}/token/'

os.makedirs(os.path.dirname(out_folder_lemma), exist_ok=True)
os.makedirs(os.path.dirname(out_folder_token), exist_ok=True)

In [ ]:
# Get sentences with lemma_POS in lemma and their corresponding token sentences
for selected_lemma in selected_lemmas:
    df_lemma_filtered = df_lemma[df_lemma['sent'].str.contains(f' {selected_lemma} ', regex=False)] # index is kept, need to separate by space to avoid partial match
    df_token_filtered = df_token.loc[df_lemma_filtered.index]

    # Save filtered DataFrames to new CSV files
    df_lemma_filtered.to_csv(f'{out_folder_lemma}/ccoha{corp_no}_{selected_lemma}.csv', index=False, header=True)
    df_token_filtered.to_csv(f'{out_folder_token}/ccoha{corp_no}_{selected_lemma}.csv', index=False, header=True)

### Re-parse with Stanza

In [ ]:
out_folder_reparsed = f'./SemEval_en_split/corpus{corp_no}/reparsed/'
os.makedirs(os.path.dirname(out_folder_reparsed), exist_ok=True)

In [ ]:
stanza.download('en')
nlp = stanza.Pipeline(
        'en',
        processors='tokenize,mwt,pos,lemma,depparse',
        use_gpu=True,
        verbose=False,
        tokenize_no_ssplit=True
    )

In [ ]:
for selected_lemma in selected_lemmas:
    selected_lemma_token_df = pd.read_csv(f'./SemEval_en_split/corpus{corp_no}/token/ccoha{corp_no}_{selected_lemma}.csv')

    reparsed_sents = []

    i = 0
    for sent in selected_lemma_token_df['sent']:
        doc = nlp(sent)
        for s in doc.sentences:
            lines = []
            lines.append(f'<s id={selected_lemma}_{i}>')
            for w in s.words:
                lines.append(
                    f"{w.text}\t{w.lemma}\t{w.upos}\t{w.id}\t{w.head}\t{w.deprel}"
                )
            lines.append("</s>")
            reparsed_sents.append("\n".join(lines))
            i += 1

    # Save reparsed sentences to file
    with open(f'{out_folder_reparsed}/ccoha{corp_no}_{selected_lemma}_reparsed.txt', 'w') as f:
        f.write("\n\n".join(reparsed_sents))

In [ ]:
# Lowercase the lemma form in the reparsed files
for corp_no in [1, 2]:
    for selected_lemma in selected_lemmas:
        reparsed_file = f'./SemEval_en_split/corpus{corp_no}/reparsed/ccoha{corp_no}_{selected_lemma}_reparsed.txt'

        with open(reparsed_file, 'r') as f:
            content = f.read()

        lines = content.split('\n')
        normalised_lines = []
        for line in lines:
            if line.startswith('<s id=') or line.startswith('</s>') or line.strip() == '':
                normalised_lines.append(line)
            else:
                parts = line.split('\t')
                if len(parts) >= 2:
                    parts[1] = parts[1].lower()
                    normalised_lines.append('\t'.join(parts))
                else:
                    normalised_lines.append(line)

        with open(reparsed_file, 'w') as f:
            f.write('\n'.join(normalised_lines))

### Check and quantify the mistmatch between Stanza reparsed and original

In [ ]:
# MAIN FUNCTION TO CHECK FOR MISMACHES

# Create 1 dictionary to save mismatch for all selected_lemmas
report_mismatches = {}

for corp_no in [1, 2]:
    report_mismatches[corp_no] = {}
    for selected_lemma in selected_lemmas:
        # Whole file statistics:
        mismatch_sent = 0
        org_miss_lemma_count = 0
        reparsed_miss_lemma_count = 0

        report_mismatches[corp_no][selected_lemma] = []
        org_parsed_file = f'./SemEval_en_split/corpus{corp_no}/lemma/ccoha{corp_no}_{selected_lemma}.csv'
        reparsed_file = f'./SemEval_en_split/corpus{corp_no}/reparsed/ccoha{corp_no}_{selected_lemma}_reparsed.txt'

        # Read org lemma file
        org_df = pd.read_csv(org_parsed_file)
        org_df['sent'] = org_df['sent'].fillna('')
        org_sents = org_df['sent'].tolist()
        org_sent_count = len(org_sents)
        pattern = rf'\b{re.escape(selected_lemma)}\b'
        org_lemma_count = org_df['sent'].str.count(pattern).sum()

        # Read reparsed file
        with open(reparsed_file, 'r') as f:
            reparsed_content = f.read()
            # Separate the sents
            reparsed_sents = reparsed_content.strip().split("\n\n")
            reparsed_sent_count = len(reparsed_sents)

        # Check no of sents
        if org_sent_count != reparsed_sent_count:
            report_mismatches[corp_no][selected_lemma].append(f"No of sentences mismatch: org={org_sent_count}, reparsed={reparsed_sent_count}")
            pass
        
        else:
            # Individual sentence check
            for i in range(org_sent_count):
                org_sent = org_sents[i]
                reparsed_sent = reparsed_sents[i]

                # Count selected_lemma occurrences in original sentence
                org_count = len(re.findall(pattern, org_sent))
                # Count selected_lemma occurrences in reparsed sentence
                stanza_lemma, stanza_pos = convert_org_stanza(selected_lemma)
                stanza_format = f'\t{stanza_lemma}\t{stanza_pos}\t'
                reparsed_count = reparsed_sent.count(stanza_format)

                # Return stanza tags for the selected_lemma
                stanza_tags = return_stanza_parsed_tags(reparsed_sent, selected_lemma)
                if org_count != reparsed_count:
                    mismatch_sent += 1
                    (report_mismatches[corp_no][selected_lemma].append(
                        f"Mismatch sentence:{i}, "
                        f"org={org_count}, "
                        f"reparsed={reparsed_count}, "
                        f"org_sent='{org_sent}, "
                        f"stanza_pos={stanza_tags}"))
                    
                    if org_count >= reparsed_count:
                        reparsed_miss_lemma_count += (org_count - reparsed_count)
                    elif org_count < reparsed_count:
                        org_miss_lemma_count += (reparsed_count - org_count)
            
            # Whole file statistics:
            if mismatch_sent != 0:
                report_mismatches[corp_no][selected_lemma].append(f"Total mismatched sentences: {mismatch_sent} ({mismatch_sent/org_sent_count*100:.2f}%)")
                report_mismatches[corp_no][selected_lemma].append(f"Total missing lemma in original file (compared to org): {org_miss_lemma_count} ({org_miss_lemma_count/ (org_lemma_count)*100:.2f}%)")
                report_mismatches[corp_no][selected_lemma].append(f"Total missing lemma in reparsed file (compared to org): {reparsed_miss_lemma_count} ({reparsed_miss_lemma_count/org_lemma_count*100:.2f}%)")

In [ ]:
report_mismatches

In [ ]:
with open('./SemEval_en_split/mismatch_report.txt', 'w') as f:
    # Write the report_mismatches dictionary to the file beautifully
    for corp_no in report_mismatches:
        f.write(f"Corpus {corp_no}:\n")
        for selected_lemma in report_mismatches[corp_no]:
            f.write(f"Lemma: {selected_lemma}\n")
            for mismatch in report_mismatches[corp_no][selected_lemma]:
                f.write(f"{mismatch}\n")
            f.write("\n")

### Fix strategy:


In [ ]:
# PROPN -> NOUN
for corp_no in [1, 2]:
    for selected_lemma in selected_lemmas:
        stanza_lemma, _ = convert_org_stanza(selected_lemma)

        reparsed_file = f'./SemEval_en_split/corpus{corp_no}/reparsed/ccoha{corp_no}_{selected_lemma}_reparsed.txt'
        content = open_txt(reparsed_file)
        new_content = replace_in_txt(content, f'\t{stanza_lemma}\tPROPN\t', f'\t{stanza_lemma}\tNOUN\t')
        save_to_txt(content, reparsed_file)

In [ ]:
# Fixing in each file
# landing -> land
# qquilt -> NOUN
# quilting -> quilt
# stroke -> VERB
# tipping -> tip, tip -> VERB
# stabbing -> stab, stab -> VERB

# chairmen -> chairman
# uilts -> quilt
# gase -> gas
# lasse -> lass
# heads -> head
# pinn -> pin
# play -> player
# rags -> rag
# record ADJ -> record NOUN
# records -> record

### Merge 2 corpus and separate into different selected_lemma folders

In [ ]:
merge_output =  Path(f'./SemEval_en_split/merged_corpus/')
os.makedirs(os.path.dirname(merge_output), exist_ok=True)
corpus_1 = Path(f'./SemEval_en_split/corpus1/reparsed/')
corpus_2 = Path(f'./SemEval_en_split/corpus2/reparsed/')
corpus_file_pattern = r'ccoha(\d+)_(.+?)_reparsed.txt'

In [ ]:
for selected_lemma in selected_lemmas:
    file1_path = corpus_1 / f'ccoha1_{selected_lemma}_reparsed.txt'
    file2_path = corpus_2 / f'ccoha2_{selected_lemma}_reparsed.txt'
    
    merged_output_folder = merge_output / selected_lemma
    corpus_1_folder = merged_output_folder / '1'
    corpus_2_folder = merged_output_folder / '2'

    os.makedirs(corpus_1_folder, exist_ok=True)
    os.makedirs(corpus_2_folder, exist_ok=True)

    # Copy file1 and file 2 to merged folder
    shutil.copy(file1_path, corpus_1_folder)
    shutil.copy(file2_path, corpus_2_folder)

## Stanza-cleaned

### Parse with Stanza

In [ ]:
stanza.download('en')
nlp = stanza.Pipeline(
        'en',
        processors='tokenize,mwt,pos,lemma,depparse',
        use_gpu=True,
        verbose=False,
        tokenize_no_ssplit=True
    )

In [ ]:
BATCH_SIZE = 100

for corp_no in [1, 2]:
    out_folder_reparsed = f'./SemEval_en_no_split/corpus{corp_no}/reparsed/'
    os.makedirs(out_folder_reparsed, exist_ok=True)

    token_df = pd.read_csv(f'./SemEval_en_no_split/corpus{corp_no}/token/ccoha{corp_no}.csv')
    sents = token_df['sent'].tolist()
    reparsed_sents = [] 

    i = 0

    for start in tqdm(range(0, len(sents), BATCH_SIZE)):
        batch = sents[start:start + BATCH_SIZE]
        
        # Batch processing
        texts = "\n\n".join(batch)
        
        doc = nlp(texts)
        for s in doc.sentences:
            lines = []
            lines.append(f'<s id={corp_no}_{i}>')
            for w in s.words:
                lines.append(
                    f"{w.text}\t{w.lemma}\t{w.upos}\t{w.id}\t{w.head}\t{w.deprel}"
                )
            lines.append("</s>")
            reparsed_sents.append("\n".join(lines))
            i += 1

    out_path = f'{out_folder_reparsed}/ccoha{corp_no}_reparsed.txt'
    # Save reparsed sentences to file
    with open(out_path, 'w', encoding = 'utf-8') as f:
        f.write("\n\n".join(reparsed_sents))

In [ ]:
# Lowercase the lemma form in the reparsed files
for corp_no in [1, 2]:
    reparsed_file = f'./SemEval_en_no_split/corpus{corp_no}/reparsed/ccoha{corp_no}_reparsed.txt'

    with open(reparsed_file, 'r') as f:
        content = f.read()

    lines = content.split('\n')
    normalised_lines = []
    for line in lines:
        if line.startswith('<s id=') or line.startswith('</s>') or line.strip() == '':
            normalised_lines.append(line)
        else:
            parts = line.split('\t')
            if len(parts) >= 2:
                parts[1] = parts[1].lower()
                normalised_lines.append('\t'.join(parts))
            else:
                normalised_lines.append(line)

    with open(reparsed_file, 'w') as f:
        f.write('\n'.join(normalised_lines))

### Minimal Fix

In [ ]:
# PROPN -> NOUN
# chairmen -> chairman
# quilts -> quilt
# gase -> gas
# lasse -> lass
# heads -> head
# pinn -> pin
# rags -> rag
# records -> record

### Copy files to a merge folder

In [ ]:
merge_output =  Path(f'./SemEval_en_no_split/merged_corpus/')
os.makedirs(merge_output, exist_ok=True)
corpus_1 = Path(f'./SemEval_en_no_split/corpus1/reparsed/')
corpus_2 = Path(f'./SemEval_en_no_split/corpus2/reparsed/')
corpus_file_pattern = r'ccoha(\d+)_reparsed.txt'

In [ ]:
file1_path = corpus_1 / f'ccoha1_reparsed.txt'
file2_path = corpus_2 / f'ccoha2_reparsed.txt'
    
merged_output_folder = merge_output
corpus_1_folder = merged_output_folder / '1'
corpus_2_folder = merged_output_folder / '2'

os.makedirs(corpus_1_folder, exist_ok=True)
os.makedirs(corpus_2_folder, exist_ok=True)

# Copy file1 and file 2 to merged folder
shutil.copy(file1_path, corpus_1_folder)
shutil.copy(file2_path, corpus_2_folder)